# Introduction to Correlation Power Analysis

This notebook records power traces and compares them with a simple prediction: the Hamming weight of the first input byte. It introduces two ways to look for a relationship between input data and measured power: grouped mean traces and Pearson correlation.

#### Learning goals
- Capture traces for different inputs
- Group traces by the first input byte's Hamming weight
- Calculate and interpret a correlation value at each sample

This is an introduction to the analysis, not a key-recovery attack. The target transforms each input byte using an S-box and a key byte, so the Hamming weight of the input alone is only a simple example model, not an accurate model of the target's intermediate value.

In [1]:
# Import tools for trace capture, array processing, statistics, and plotting.
import os

import numpy as np
import plotly.graph_objects as pgo
from scipy.stats import pearsonr
from cwtoolbox import CaptureDevice

## Prepare the target

The next cell creates a connection to a CW-Lite XMEGA capture device, compiles the S-box example from `lecture_3`, and loads the compiled program onto the target. The source path is relative to this notebook, so run the notebook with `lecture_4` as the working directory. A connected and configured capture device is required.

In [2]:
# Connect to the target board and compile the S-box example for it.
capture_device = CaptureDevice.create("CWLITEXMEGA")
capture_device.compile(file=os.path.abspath("../lecture_3/sbox_lookup.c"))
# Load the compiled program onto the target.
capture_device.flash()

XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


## Record traces

The capture creates 500 measurements. For each one, it sends a new random 16-byte input and records 2,240 samples of the power trace. The fixed random seed makes the generated input sequence repeatable; it does not make the physical measurements identical.

The returned `data` object contains an `input` array and a `trace` array. Row `i` in each array belongs to the same measurement: `data["input"][i]` is the input that produced `data["trace"][i]`. In the shape output below, the first dimension is the number of measurements; the second dimension is the number of input bytes or trace samples, respectively.

In [3]:
# Capture 500 traces, each containing 2,240 samples.
number_traces = 500
number_samples = 140 * 16
# A fixed seed makes the sequence of random inputs repeatable.
rng = np.random.default_rng(42)

# Send a new random 16-byte input for each trace and keep both results.
data = capture_device.capture(
    number_of_traces=number_traces,
    number_of_samples=number_samples,
    input=lambda _: rng.integers(0, 256, size=16, dtype=np.uint8),
)

100%|██████████| 500/500 [00:04<00:00, 111.49it/s]


In [4]:
# Check the number of captured inputs and the size of the trace matrix.
print("Input array shape:", data["input"].shape)
print("Trace array shape:", data["trace"].shape)

Input array shape: (500, 16)
Trace array shape: (500, 2240)


## Group traces by Hamming weight

A byte's **Hamming weight** is the number of bits in it that are `1`. For example, `00000101` has Hamming weight 2. A byte therefore has a Hamming weight from 0 to 8. The lookup table in the code maps each possible byte to its Hamming weight.

### `plot_hamming_weight`
This function looks only at the first byte of each input. It groups traces according to that byte's Hamming weight, averages the traces in each group sample by sample, and plots the group averages. Some weights may not occur in the captured inputs, so those groups are omitted.

In [5]:
# Precompute the Hamming weight for every possible byte value (0-255).
HW_LUT = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)


def plot_hamming_weight(x, traces):
    # Convert inputs to arrays and check that each input matches one trace.
    x = np.asarray(x)
    traces = np.asarray(traces)
    if x.ndim != 1 or traces.ndim != 2 or len(x) != len(traces):
        raise ValueError("x must contain one byte per two-dimensional trace row")

    # Assign each trace to the Hamming-weight group of its input byte.
    hamming_weights = HW_LUT[x]
    fig = pgo.Figure()
    for weight in range(9):
        # Average all traces in this group; skip groups with no measurements.
        group = traces[hamming_weights == weight]
        if len(group):
            fig.add_trace(
                pgo.Scatter(y=group.mean(axis=0), name=f"HW {weight}")
            )

    # Label the plot so the sample axis and measured values are clear.
    fig.update_layout(
        title="Mean traces grouped by input Hamming weight",
        xaxis_title="Sample",
        yaxis_title="Mean trace value",
    )
    fig.show()

### How to read the mean-trace plot

Each line is the **average trace** for one group of measurements. For example, `HW 3` averages the traces whose first input byte contains three `1` bits. At any sample position, the line's height is the average measured value for that group at that sample.

- **Separated averages** mean the measured signal differs between Hamming-weight groups at that sample. This may be evidence that activity related to the input affects the measurement there.
- **Overlapping averages** mean this plot shows no clear difference between the groups at that sample. They do not prove that the input is unused; noise or other activity can hide a difference.

Treat the plot as a visual clue, not proof of which instruction caused a signal. Group averages can hide small or inconsistent effects, and unrelated data that changes with the input can also produce differences. The correlation plot that follows gives a numerical measure of the linear relationship.

The model here is the Hamming weight of the input byte. The target actually looks up an S-box value using the input combined with a key byte, so a full CPA attack would need to model that intermediate value instead.

In [6]:
# Compare mean traces grouped by the first byte of each input.
plot_hamming_weight(data["input"][:, 0], data["trace"])

## Correlation trace

Pearson correlation measures how strongly two lists have a **linear relationship**. Here, we compare predicted values from a simple power model with measured values from the traces, one sample position at a time.

### `corr_hamming_weight`
This function returns one Pearson correlation coefficient for each sample position. A coefficient near `+1` indicates a strong positive linear relationship, near `-1` a strong negative relationship, and near `0` little linear relationship. A value near zero does not rule out a non-linear relationship.

### Why calculate correlation at every sample?

Suppose we have **n traces**, each containing **s samples**. Put the measurements in a matrix `T` with `n` rows and `s` columns:

- Row `i` is trace `i`.
- Column `t` contains sample `t` from every trace.
- `T[i, t]` is the measured value in trace `i` at sample position `t`.

Let `x[i]` be the first input byte used for trace `i`. Our simple model predicts its Hamming weight, `h[i] = HW(x[i])`. This gives one predicted value for each trace, collected in the vector `h`.

At a particular sample position `t`, compare the `n` predicted values with the `n` measurements in column `t`:

- Predicted: `h[0], h[1], ..., h[n-1]`
- Measured: `T[0, t], T[1, t], ..., T[n-1, t]`

If traces are aligned, sample position `t` corresponds approximately to the same moment in the program in every trace. If the model describes data affecting the measurement at that moment, the two lists may vary together.

The code computes `r[t] = pearsonr(h, T[:, t])[0]` for each `t` from `0` to `s-1`. The `[0]` selects the correlation coefficient from SciPy's result. The collection of these `s` coefficients is the **correlation trace**; plotting it shows where the model has the strongest linear relationship with the measurements.

If measurements were a positive linear function of the predicted values with no noise, the coefficient would be `+1`. Noise, other chip activity, imperfect alignment, and an inaccurate model usually reduce its magnitude. A strong coefficient is evidence of a linear relationship, not proof of which operation caused it. A weak coefficient can mean the model is inaccurate or the signal is noisy; it does not prove that the value was never processed.

For this demonstration, the model is `HW(input byte)`. The target processes `SBOX[input byte XOR key byte]`, so a realistic CPA attack would instead predict the Hamming weight of that intermediate value for each key guess.

In [7]:
def corr_hamming_weight(x, traces):
    # Convert inputs to arrays and check that each input matches one trace.
    x = np.asarray(x)
    traces = np.asarray(traces)
    if x.ndim != 1 or traces.ndim != 2 or len(x) != len(traces):
        raise ValueError("x must contain one byte per two-dimensional trace row")

    # Use input Hamming weights as the predicted leakage for each trace.
    hamming_weights = HW_LUT[x]
    corr_trace = np.empty(traces.shape[1], dtype=float)
    for sample in range(traces.shape[1]):
        # Compare the model with measurements at this sample position.
        corr_trace[sample] = pearsonr(hamming_weights, traces[:, sample])[0]

    # Return one correlation value for each sample in the traces.
    return corr_trace

In [8]:
# Calculate correlation over time for the first input byte's Hamming weight.
corr = corr_hamming_weight(data["input"][:, 0], data["trace"])

# Plot one correlation coefficient for each trace sample.
fig = pgo.Figure(pgo.Scatter(y=corr, name="Input byte 0"))
fig.update_layout(
    title="Correlation of input Hamming weight with each trace sample",
    xaxis_title="Sample",
    yaxis_title="Pearson correlation coefficient",
    yaxis_range=[-1, 1],
)
fig.show()

C:\Users\probstmatthi\AppData\Local\Temp\ipykernel_38896\3765392051.py:13: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr_trace[sample] = pearsonr(hamming_weights, traces[:, sample])[0]


## Further exploration

The correlation plot above uses the input byte's Hamming weight as its prediction. In a CPA attack, a prediction is made for a specific intermediate value under each key guess, then compared with the traces.

You can apply the same general idea of comparing predictions with measurements to other controlled lab examples, such as the password-check program from lecture 2. However, `corr_hamming_weight` only tests whether `HW(input)` is linearly related to each trace sample. It does **not** automatically identify when a password character is compared or reveal a matching character. The model and the interpretation must fit the program being studied.